In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    缩放点积注意力
    """
    embed_size = Q.size(-1)
    scores = Q @ K.transpose(-1, -2) / math.sqrt(embed_size)
    
    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)

    output = attention_weights @ V
    return output, attention_weights

## 缩放点积注意力
#### 取Q矩阵最后一维（特征维度）
#### 计算注意力分数
#### masked_fill（condition， value）将mask矩阵等于零的地方（掩码），置为负无穷（softmax后为零）
#### 对最后一维做归一化

In [12]:
class Attention(nn.Module):
    def __init__(self, embed_size):
        super(self, Attention).__init__()
        self.embed_size = embed_size

        self.w_q = nn.Linear(embed_size, embed_size)
        self.w_k = nn.Linear(embed_size, embed_size)
        self.w_v = nn.Linear(embed_size, embed_size)
    
    def forward(self, q, k, v, mask=None):
        Q = self.w_q(q)
        K = self.w_k(k)
        V = self.w_v(v)

        out, attention_weights = scaled_dot_product_attention(Q, K, V, mask)

        return out, attention_weights
         
    

## 单头注意力

In [13]:
scores = (Q @ K.tranpose(-1, -2)) / math.sqrt(embed_size)
if mask is not None:
    scores = scores.masked_fill(mask==0, float('-inf'))

NameError: name 'Q' is not defined

## 掩码
#### 掩码机制并不是直接在截断输入序列，也不是在算分数的时候就排除不应该看到的位置，因为看到也没有关系，不会影响与其他位置的分数
#### 掩码包括了填充掩码和未来掩码
#### 因为pytorch张量的广播机制，掩码矩阵可以只设置成(seq_len, head_dim)，batch_size和num_head可以自动广播

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()
        self.attention = Attention(embed_size)
    
    def forward(self, x, mask=None):
        out, attention_weights = self.attention(x, x, x, mask)
        return out, attention_weights

## 自注意力
#### 需要把Attention写为一个永久属性，这样才能实现参数优化，如果只是Attention(x, x, x, mask)，相当于每次都给一个新的Attention，无法参数更新

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, embed_size):
        super(CrossAttention, self).__init__()
        self.attention = Attention(embed_size)
    def forward(self, q, kv, mask-None):
        out, attention_weights = self.attention(q, kv, kv, mask)
        return out, attention_weights

## Transformer中的交叉注意力
#### 解码器提供Q， K和V来自编码器：用已经生成的seq决定生成下一个词时应该“注意”哪些信息

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadAttention, self).__init__()

        self.embed_size = embed_size
        self.num_heads = num_heads

        self.w_q = nn.ModuleList([nn.Linear(embed_size, embed_size) for _ in range(num_heads)])
        self.w_k = nn.ModuleList([nn.Linear(embed_size, embed_size) for _ in range(num_heads)])
        self.w_v = nn.ModuleList([nn.Linear(embed_size, embed_size) for _ in range(num_heads)])

        self.fc_out = nn.Linear(num_heads * embed_size, embed_size)
    
    def forward(self, q, k, v, mask=None):
        batch_size = q.shape[0]
        multi_head_outputs = []

        for i in range(self.num_heads):
            Q = self.w_q[i](q)
            K = self.w_k[i](k)
            V = self.w_v[i](v)

            scaled_attention, _ = scaled_dot_product_attention(Q, K, V, mask)
            multi_head_outputs.append(scaled_attention)
        
        concat_output = torch.cat(multi_head_outputs, dim=-1)

        out = self.fc_out(concat_output)

        return out

def scaled_dot_product_attention(Q, K, V, mask=None):
    embed_size = Q.shape[-1]
    scores = (Q @ K.transpose(-1, -2)) /math.sqrt(embed_size)

    if mask is not None:
        scores = torch.masked_fill(mask==0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    out = attention_weights @ V

    return out, attention_weights
    


## 多头注意力机制的基本实现
#### 通常需要将每个头的输入维度调为embed_size // num_heads 确保和单头的参数量一样
#### 多个头拼接在一起后，还需要过一个线性层，来使不同头之间的信息融合。

In [ ]:
class MultiAttention(nn.Module):
    def __init__(self, embed_size, num_head):
        super(MultiAttention, self).__init__()

        assert embed_size % num_head == 0

        self.embed_size = embed_size
        self.num_head = num_head
        head_dim = embed_size / num_head
        self.head_dim = head_dim

        self.w_q = nn.Linear(embed_size, embed_size)
        self.w_k = nn.Linear(embed_size, embed_size)
        self.w_v = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, q, k, v, mask):
        batch_size = q.shape[0]
        seq_len_q = q.shape[1]
        seq_len_k = k.shape[1]
        Q = self.w_q(q).reshape(batch_size, seq_len_q, self.num_head, self.head_dim).transpose(1, 2)
        K = self.w_k(k).reshape(batch_size, seq_len_k, self.num_head, self.head_dim).transpose(1, 2)
        V = self.w_v(v).reshape(batch_size, seq_len_k, self.num_head, self.head_dim).transpose(1, 2)

        scaled_attention, _ = scaled_dot_product_attention(Q, K, V, mask)

        concat_out = scaled_attention.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_size)

        out = self.fc_out(concat_out)

        return out


## 多头注意力的标准实现
#### Transformer中是先把整个词向量线性变换，再分成多个头
#### 一个torch张量在内存中连续存储，reshape的时候会先用seq_len = 0 维度的输出填充新形状的张量，然后用seq_len = 1 ......所以reshape的时候需要先按seq_len分，然后按头分，最后再转置。如果直接先按头分，就无法得到想要的结果
#### 多个头的张量信息拼接时，需要先把维度换回去，然后重塑维度（因为view要求被重塑的张量在内存中连续存储，所有需要先contiguous调整张量的存储方式）。

In [ ]:
class PositionwiseFeedforward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedforward, self).__init__()

        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w_2(F.relu(self.w_1(x)))

## 逐位置前馈层
#### 对每个位置的词向量分别使用同一个前馈神经网络进行变换。相当于一句话的每个词向量看作一个批次，进行批量梯度下降来更新前馈层的参数。
#### 具体实现就是中间有一个relu激活函数的两个线性层

In [ ]:
class ResidualConnection(nn.Module):
    def __init__(self, dropout=0.1):
        super(ResidualConnection, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, sublayer):

        return x + self.dropout(sublayer(x))

## 残差连接
#### dropout直接使sublayer输出的张量一部分为零，可以防止过拟合，提升模型泛化能力（模型需要更多地发掘数据特征，因为已发掘的特征可能会被置零）

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, feature_size, epsilon=1e-9):
        super(LayerNorm, self).__init__()

        self.gamma = nn.parameter(torch.ones(feature_size))
        self.beta = nn.parameter(torch.zeros(feature_size))
        self.epsilon = epsilon

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)

        return self.gamma * (x - mean) / (std + self.epsilon) + self.beta

## 归一化
#### nn.parameters：构建一个可学习张量
#### 归一化后，数据均值强制为零，方差强制为1，α和β可以引入一定的不确定性，提升模型的能力（有可能归一化会使效果变差）
#### x.mean(dim=-1, keepdim=True) keepdim可以使均值张量的维度与之前相同

In [ ]:
class SublayerConnection(nn.Module):
    def __init__(self, feature_size, dropout=0.1, epsilon=1e-9):
        super(SublayerConnection, self).__init__()

        self.residual = ResidualConnection(dropout)
        self.norm = LayerNorm(feature_size, epsilon)

    def forwaerd(self, x, sublayer):
        return self.norm(self.residual(x, sublayer))

## Add+Norm
#### Post-Norm，即在残差链接后做归一化 这是Transformer论文中的做法
#### 目前应用更广泛的是pre_Norm，因为post-norm存在训练不稳定的情况：
残差主干道上的障碍：在 Post-Norm 中，LayerNorm 层位于残差连接的主干道上。当模型的层数非常深时，从顶层传向底层的梯度在反向传播时，必须穿过每一个 LayerNorm 层。LayerNorm 中可学习的缩放参数 𝛾会不断地累积乘以梯度，这可能导致梯度在深层网络中爆炸或消失，使得非常深（例如超过12层）的 Transformer 难以训练。

训练初期不稳定：如前所述，训练初期 SubLayer(x) 的输出尺度很大，导致 x+SubLayer(x) 的尺度也很大。虽然 LayerNorm 会将其拉回，但在反向传播时，这些巨大的中间值会产生巨大的梯度，导致训练过程非常不稳定。为了解决这个问题，Post-Norm 的训练严重依赖学习率预热（Learning Rate Warmup），需要用一个很小的学习率开始，慢慢增大学习率，让模型先“适应”一下，否则很容易直接发散。

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super(Embeddings, self).__init__()

        self.embed = nn.Embedding(vocab_size, d_model)
        self.scale_factor = math.sqrt(d_model)

    def forward(self, x):
        return self.embed(x) * self.scale_factor

## 嵌入层
#### 需要对结果使用scale_factor进行缩放，用来调整位置编码和语义信息的权重

In [ ]:
logits = []
target = []
nn.CrossEntropyLoss(logits, target)
log_probs = F.log_softmax(logits)
nn.NLLLoss(log_probs, target)

## 损失函数
#### nn.CrossEntropyLoss输入logits，损失函数内包含softmax
#### nn.NLLLoss，损失函数内无softmax

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000 ):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        # 位置索引
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 增加一个维度，方便后续与输入相加，形状变为 (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)
    
    def forward(self, x):

        x = x + self.pe[:, :x.size(1), :]

        return self.dropout(x)

NameError: name 'nn' is not defined

## 位置编码
#### position = torch.arange(0, max_len).unsqueeze(1) 计算每个时间步的位置索引
#### unsequeeze：给指定位置添加一个维度

In [ ]:
class SourceEmbedding(nn.Module):
    def __init__(self, src_vocab, d_model, dropout=0.1):
        super(SourceEmbedding, self).__init__()
        self.emded = Embeddings(src_vocab, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout)
    
    def forward(self, x):
        x = self.emded(x)
        return self.positional_encoding(x)

## 编码器输入

In [ ]:
class TargetEmbedding(nn.Module):
    def __init__(self, tgt_vocab_size, d_model, dropout=0.1):
        super(TargetEmbedding, self).__init__()
        self.embed = Embeddings(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout)
    
    def forward(self, x):
        x = self.embed(x)
        return self.positional_encoding(x)

## 解码器输入
#### 编码器需要掩码，同时需要保证模型预测每一个位置的词时，有位置信息但没有改位置的词，尤其是第一个词
#### 通过添加开始和结束标识实现，tgt = "< sos > I love NLP < eos >"
#### 输入：< sos > I love NLP 输出：I love NLP < eos >
#### 右移操作通常不是在 Transformer 模型内部处理的，而是在数据传入模型之前的预处理阶段

In [ ]:
def create_padding_mask(seq, pad_token_id=0):
    mask = (seq != pad_token_id).unsqueeze(1).unsqueeze(2)
    return mask

## 填充掩码
#### 处理不等长的输入序列时，需要使用padding来填充短序列。因此，计算注意力时填充部分不能对结果产生影响，即为零的地方，q和k的匹配度应该是0
#### 为方便与注意力分数矩阵的维度(batch_size, num_heads, seq_len_q, seq_len_k)配合，掩码矩阵写成(batch_size, 1, 1, seq_len_tgt)

In [ ]:
def create_ahead_mask(size):
    mask = torch.tril(torch.ones(size, size)).type(torch.bool)
    return mask

## 未来信息掩码
#### 用于解码器中屏蔽未来位置

In [ ]:
def create_decoder_mask(tgt_seq, pad_token_id=0):
    padding_mask = create_padding_mask(tgt_seq, pad_token_id)
    look_ahead_mask = create_ahead_mask(tgt_seq.size(1)).to(tgt_seq.device)

    combined_mask = look_ahead_mask.unsqueeze(0) & padding_mask
    return combined_mask

## 综合掩码